In [0]:
silver = spark.table("workspace.api_logs_schema.silver_api_logs_clean")

In [0]:
display(silver.limit(3))

request_id,event_time,service,endpoint,method,status_code,latency_ms,bytes_in,bytes_out,client_type,region,host
req_20260423_000001,2026-04-23T10:57:02Z,auth-api,/v1/login,POST,201,119,173,38589,mobile,ap-south-1,app-04
req_20260423_000002,2026-04-23T10:02:27Z,search-api,/v1/search,GET,201,154,1959,25404,desktop,us-east-1,app-02
req_20260423_000003,2026-04-23T10:09:19Z,auth-api,/v1/login,PUT,500,174,949,48629,mobile,eu-west-1,app-05


In [0]:
dim = spark.read.csv(
    "/Volumes/workspace/api_logs_schema/api_logs_volume/service_catalog.csv",
    header=True,
    inferSchema=True
)

display(dim)

service,team,sla_ms,criticality
auth-api,platform,150,high
docs-api,content,300,medium
search-api,search,250,high
notifications-api,engagement,400,low


In [0]:
from pyspark.sql.functions import to_timestamp, to_date, hour, col

In [0]:
silver_enriched = silver\
    .withColumn("event_ts",to_timestamp("event_time"))\
    .withColumn("event_date",to_date("event_ts"))\
    .withColumn("event_hour",hour("event_ts"))\
    .withColumn("is_client_error",((col("status_code")>=400) & (col("status_code")<500)).cast("int"))\
    .withColumn("is_server_error",((col("status_code")>=500)).cast("int"))

In [0]:
display(silver_enriched.limit(2))

request_id,event_time,service,endpoint,method,status_code,latency_ms,bytes_in,bytes_out,client_type,region,host,event_ts,event_date,event_hour,is_client_error,is_server_error
req_20260423_000001,2026-04-23T10:57:02Z,auth-api,/v1/login,POST,201,119,173,38589,mobile,ap-south-1,app-04,2026-04-23T10:57:02.000Z,2026-04-23,10,0,0
req_20260423_000002,2026-04-23T10:02:27Z,search-api,/v1/search,GET,201,154,1959,25404,desktop,us-east-1,app-02,2026-04-23T10:02:27.000Z,2026-04-23,10,0,0


In [0]:
joined = silver_enriched.join(dim, on="service",how = "left")

In [0]:
display(joined.limit(3))

service,request_id,event_time,endpoint,method,status_code,latency_ms,bytes_in,bytes_out,client_type,region,host,event_ts,event_date,event_hour,is_client_error,is_server_error,team,sla_ms,criticality
auth-api,req_20260423_000001,2026-04-23T10:57:02Z,/v1/login,POST,201,119,173,38589,mobile,ap-south-1,app-04,2026-04-23T10:57:02.000Z,2026-04-23,10,0,0,platform,150,high
search-api,req_20260423_000002,2026-04-23T10:02:27Z,/v1/search,GET,201,154,1959,25404,desktop,us-east-1,app-02,2026-04-23T10:02:27.000Z,2026-04-23,10,0,0,search,250,high
auth-api,req_20260423_000003,2026-04-23T10:09:19Z,/v1/login,PUT,500,174,949,48629,mobile,eu-west-1,app-05,2026-04-23T10:09:19.000Z,2026-04-23,10,0,1,platform,150,high


In [0]:
from pyspark.sql.functions import when

In [0]:
joined = joined.withColumn(
    "sla_breach",
    when(col("latency_ms")>col("sla_ms"),1).otherwise(0)
)
display(joined.limit(2))

service,request_id,event_time,endpoint,method,status_code,latency_ms,bytes_in,bytes_out,client_type,region,host,event_ts,event_date,event_hour,is_client_error,is_server_error,team,sla_ms,criticality,sla_breach
auth-api,req_20260423_000001,2026-04-23T10:57:02Z,/v1/login,POST,201,119,173,38589,mobile,ap-south-1,app-04,2026-04-23T10:57:02.000Z,2026-04-23,10,0,0,platform,150,high,0
search-api,req_20260423_000002,2026-04-23T10:02:27Z,/v1/search,GET,201,154,1959,25404,desktop,us-east-1,app-02,2026-04-23T10:02:27.000Z,2026-04-23,10,0,0,search,250,high,0


In [0]:
from pyspark.sql.functions import sum, avg, count, expr

In [0]:
from pyspark.sql.functions import count, sum, avg, expr, col

gold = joined.groupBy("event_date", "service", "team", "criticality") \
    .agg(
        count("*").alias("total_requests"),
        sum("is_client_error").alias("client_error_requests"),
        sum("is_server_error").alias("server_error_requests"),
        avg("latency_ms").alias("avg_latency_ms"),
        expr("percentile(latency_ms, 0.95)").alias("p95_latency_ms"),
        sum("sla_breach").alias("sla_breach_requests"),
        sum("bytes_out").alias("total_bytes_out")
    ) \
    .withColumn(
        "error_rate",
        (col("client_error_requests") + col("server_error_requests")) / col("total_requests")
    )

display(gold)

event_date,service,team,criticality,total_requests,client_error_requests,server_error_requests,avg_latency_ms,p95_latency_ms,sla_breach_requests,total_bytes_out,error_rate
2026-04-24,docs-api,content,medium,78,10,8,115.07692307692308,182.74999999999994,0,2165831,0.23076923076923078
2026-04-24,auth-api,platform,high,71,12,7,133.49295774647888,216.0,26,1861531,0.2676056338028169
2026-04-23,auth-api,platform,high,56,10,5,115.67857142857143,180.75,10,1318915,0.26785714285714285
2026-04-24,notifications-api,engagement,low,74,10,6,120.91891891891892,208.35,0,1682738,0.21621621621621623
2026-04-23,search-api,search,high,56,8,5,111.10714285714286,204.25,0,1505032,0.23214285714285715
2026-04-24,search-api,search,high,76,12,8,109.05263157894737,199.0,0,1719335,0.2631578947368421
2026-04-23,notifications-api,engagement,low,73,9,8,117.01369863013699,194.99999999999994,0,1919939,0.2328767123287671
2026-04-23,docs-api,content,medium,65,12,9,117.96923076923076,195.0,0,1758461,0.3230769230769231


In [0]:
gold.write\
    .format("delta")\
        .mode("overwrite")\
            .saveAsTable("workspace.api_logs_schema.gold_service_daily_kpis")

In [0]:
print("Bronze rows:", spark.table("workspace.api_logs_schema.bronze_api_logs_raw").count())
print("Silver clean rows:", spark.table("workspace.api_logs_schema.silver_api_logs_clean").count())
print("Gold rows:", spark.table("workspace.api_logs_schema.gold_service_daily_kpis").count())

Bronze rows: 884
Silver clean rows: 549
Gold rows: 8
